In [ ]:
import pandas as pd

# データの読み込み
train_path = "../data/train.csv"
train_df = pd.read_csv(train_path)

# 確認対象のカテゴリ変数
categorical_cols = ["TypeofContact", "Occupation", "Gender", "ProductPitched", "Designation", "customer_info"]

# 各列のユニーク値と出現頻度を確認する関数
def check_text_variations(df, columns):
    for col in columns:
        print(f"\n=== {col} のユニーク値と出現頻度 ===")
        # ユニーク値とそのカウントを取得
        value_counts = df[col].value_counts(dropna=False)  # NaNも含める
        print(value_counts)
        print(f"ユニーク値の数: {len(value_counts)}")
        print("ユニーク値リスト:")
        print(list(value_counts.index))

# カテゴリ変数の確認を実行
check_text_variations(train_df, categorical_cols)

# 必要に応じて特定の列を詳細に確認
# 例: ProductPitched のみをさらに詳しく見る
print("\n=== ProductPitched の詳細確認 ===")
product_pitched_values = train_df["ProductPitched"].value_counts(dropna=False)
for value, count in product_pitched_values.items():
    print(f"値: {value}, 出現回数: {count}")


=== TypeofContact のユニーク値と出現頻度 ===
TypeofContact
Self Enquiry       2194
Company Invited    1289
NaN                   6
Name: count, dtype: int64
ユニーク値の数: 3
ユニーク値リスト:
['Self Enquiry', 'Company Invited', nan]

=== Occupation のユニーク値と出現頻度 ===
Occupation
Small Business    1727
Salaried          1401
Large Business     361
Name: count, dtype: int64
ユニーク値の数: 3
ユニーク値リスト:
['Small Business', 'Salaried', 'Large Business']

=== Gender のユニーク値と出現頻度 ===
Gender
Male       1226
Female      739
male        468
female      266
MALE        183
Ｍａｌｅ        126
Fe Male     109
FEMALE       99
Ｆｅｍａｌｅ       89
ｍａｌｅ         43
ｆｅｍａｌｅ       36
fe male      29
FE MALE      23
ＭＡＬＥ         18
Ｆｅ　Ｍａｌｅ      15
ＦＥＭＡＬＥ       13
ｆｅ　ｍａｌｅ       6
ＦＥ　ＭＡＬＥ       1
Name: count, dtype: int64
ユニーク値の数: 18
ユニーク値リスト:
['Male', 'Female', 'male', 'female', 'MALE', 'Ｍａｌｅ', 'Fe Male', 'FEMALE', 'Ｆｅｍａｌｅ', 'ｍａｌｅ', 'ｆｅｍａｌｅ', 'fe male', 'FE MALE', 'ＭＡＬＥ', 'Ｆｅ\u3000Ｍａｌｅ', 'ＦＥＭＡＬＥ', 'ｆｅ\u3000ｍａｌｅ', 'ＦＥ\u3000ＭＡＬＥ']

=== ProductPitched のユ

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
import re
import unicodedata

# カテゴリ変数のクリーニング関数
def clean_categorical_column(df, column):
    def clean_text(value):
        if pd.isna(value):
            return value
        value = str(value)
        # 全角を半角に変換
        value = unicodedata.normalize("NFKC", value)
        # スペースやタブを「,」に統一
        value = value.replace(" ", ",").replace("\t", ",").replace("\n", ",")
        # 小文字化
        value = value.lower()
        return value
    df[column] = df[column].apply(clean_text)
    return df

# 1. データの読み込み
train_path = "../data/train.csv"
test_path = "../data/test.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# 2. カテゴリ変数のクリーニングを最初に適用
categorical_cols = ["TypeofContact", "Occupation", "Gender", "ProductPitched", "Designation"]
for col in categorical_cols:
    train_df = clean_categorical_column(train_df, col)
    test_df = clean_categorical_column(test_df, col)

def normalize_text(text):
    # 全角を半角に変換
    text = text.translate(str.maketrans(
        '０１２３４５６７８９ＡＢＣＤＥＦＧＨＩＪＫＬＭＮＯＰＱＲＳＴＵＶＷＸＹＺａｂｃｄｅｆｇｈｉｊｋｌｍｎｏｐｑｒｓｔｕｖｗｘｙｚ',
        '0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz'
    ))
    # 小文字に変換
    text = text.lower()
    # 空白を削除
    text = text.replace('　', '')
    text = text.replace(' ', '')
    if text == 'male':
      text = 1
    elif text == 'female':
      text = 0
    else:
      print(f"Unknown text: {text}")
      text = 0
    return text

train_df['Gender'] = train_df['Gender'].apply(normalize_text)
test_df['Gender'] = test_df['Gender'].apply(normalize_text)


def extract_and_convert_age(age_str):
    if age_str is None:
      return None
    if isinstance(age_str, float) and np.isnan(age_str):
      return None
    age_str = unicodedata.normalize('NFKC', age_str).lower()

    # 数字部分を抽出
    numbers = re.findall(r'\d+', age_str)
    if numbers:
        result = int(numbers[0])
        result = max(18, result) # 10代の場合10になるが、あり得ないので18でクリップ
        # if '代' in age_str: # 逆に下がる
        #     result += 5
        return result

    # 漢数字をアラビア数字に変換
    kanji_digits = {'一': 1, '二': 2, '三': 3, '四': 4, '五': 5, '六': 6, '七': 7, '八': 8, '九': 9, '十': 10, '百': 100, '千': 1000, '万': 10000}
    arabic_number = 0
    digit = 0
    for char in reversed(age_str):
      if char in kanji_digits:
        if kanji_digits[char] >= 10:
          digit = kanji_digits[char]
        else:
          arabic_number += kanji_digits[char] * digit if digit else kanji_digits[char]
          digit = 0

    if arabic_number:
        return arabic_number

    print(f"Failed to extract:{age_str}")
    return None  # 数字が抽出できない場合はNaNを返す


# train['Age']を更新
train_df['Age'] = train_df['Age'].apply(extract_and_convert_age)
test_df['Age'] = test_df['Age'].apply(extract_and_convert_age)


def extract_number_from_duration(duration_str):
    if duration_str is None:
      return None
    if isinstance(duration_str, float) and np.isnan(duration_str):
      return None
    m = re.search(r'(\d+)', duration_str)
    if m:
        value = int(m.group(1))
        if '分' in duration_str:
            value *= 60
        return value // 60 # 見やすさのため分単位に。精度は変わらず
    else:
        return None

train_df['DurationOfPitch'] = train_df['DurationOfPitch'].apply(extract_number_from_duration)
test_df['DurationOfPitch'] = test_df['DurationOfPitch'].apply(extract_number_from_duration)

train_df.loc[train_df['NumberOfFollowups'] >= 100, 'NumberOfFollowups'] = train_df.loc[train_df['NumberOfFollowups'] >= 100, 'NumberOfFollowups'] / 100
test_df.loc[test_df['NumberOfFollowups'] >= 100, 'NumberOfFollowups'] = test_df.loc[test_df['NumberOfFollowups'] >= 100, 'NumberOfFollowups'] / 100



def extract_number_of_trips(trips_str):
    """
    'NumberOfTrips'から数値を抽出し、必要に応じて倍率を適用して返します。
    """
    if trips_str is None:
      return None
    if isinstance(trips_str, float) and np.isnan(trips_str):
      return None
    try:
        num = int(re.search(r'\d+', trips_str).group(0))
        if "四半期" in str(trips_str):
            return num * 4
        elif "半年" in str(trips_str):
            return num * 2
        else:
            return num
    except AttributeError:
        return None  # 数字が抽出できない場合はNaNを返す


# 'NumberOfTrips' 列を更新
train_df['NumberOfTrips'] = train_df['NumberOfTrips'].apply(extract_number_of_trips)
test_df['NumberOfTrips'] = test_df['NumberOfTrips'].apply(extract_number_of_trips)

def extract_income(income_str):
    """
    MonthlyIncomeから数値を抽出し、'万'が含まれる場合は10000倍して返します。
    """
    if income_str is None:
      return None
    if isinstance(income_str, float) and np.isnan(income_str):
      return None
    try:
        match = re.search(r'(\d+(?:\.\d+)?)', income_str)
        if match:
            income = float(match.group(1))
            if '万' in str(income_str):
                income *= 10000
            return income
        else:
            return None  # 数字が抽出できない場合はNaNを返す
    except Exception as e:
        print(f"Error processing income string '{income_str}': {e}")
        return None

# 'MonthlyIncome' 列を更新
train_df['MonthlyIncome'] = train_df['MonthlyIncome'].apply(extract_income)
test_df['MonthlyIncome'] = test_df['MonthlyIncome'].apply(extract_income)



def normalize_and_transform_product_pitched(text):
    normalized_text = unicodedata.normalize('NFKC', text).lower()
    length = len(normalized_text)

    if length == 4: # king
        return 5
    elif length == 5: # basic
        return 1
    elif length == 6: # deluxe
        return 3
    elif length == 8: # standard
        return 2
    elif length == 12: # super deluxe
        return 4
    else:
        return 0

train_df['ProductPitched'] = train_df['ProductPitched'].apply(normalize_and_transform_product_pitched)
test_df['ProductPitched'] = test_df['ProductPitched'].apply(normalize_and_transform_product_pitched)



def normalize_and_transform_product_pitched(text):
    normalized_text = unicodedata.normalize('NFKC', text).lower()
    length = len(normalized_text)

    if length == 2: # VP
        return 5
    elif length == 3: # AVP
        return 4
    elif length == 9: # executive
        return 3
    elif length == 14: # senior manager
        return 2
    elif length == 7: # manager
        return 1
    else:
        return 0

train_df['Designation'] = train_df['Designation'].apply(normalize_and_transform_product_pitched)
test_df['Designation'] = test_df['Designation'].apply(normalize_and_transform_product_pitched)



def convert_list_to_string(input_list):
  """Converts a list to a string, handling potential None values."""
  if input_list is None:
    return ""

  return ','.join(map(str, input_list))  # Convert list elements to strings


# # 4. 欠損値の処理
# for col in ["Age", "DurationOfPitch", "NumberOfTrips", "MonthlyIncome", "NumberOfFollowups", "PreferredPropertyStar"]:
#     train_df[col].fillna(train_df[col].median(), inplace=True)
#     test_df[col].fillna(train_df[col].median(), inplace=True)

# for col in categorical_cols:
#     train_df[col].fillna(train_df[col].mode()[0], inplace=True)
#     test_df[col].fillna(train_df[col].mode()[0], inplace=True)


def split_customer_info(text):
  """Splits customer information string into three parts."""
  if pd.isna(text):
    return [None] * 3

  # Split by whitespace, newline, commas, and slashes
  parts = re.split(r'[、/／,\s\n]+', text)

  if len(parts) == 4:
    parts[2] = parts[2] + parts[3]
    parts = parts[:3]

  # Return the first three parts or pad with NaNs if fewer than three parts
  return parts

# Split the list into three separate columns
train_df[['Marriage', 'Own_Car', 'Children']] = pd.DataFrame(train_df['customer_info'].apply(split_customer_info).tolist(), index= train_df.index)
test_df[['Marriage', 'Own_Car', 'Children']] = pd.DataFrame(test_df['customer_info'].apply(split_customer_info).tolist(), index= test_df.index)

def normalize_marriage_status(status):
    if pd.isna(status):
        return None
    status = unicodedata.normalize('NFKC', status).lower()
    if "離婚" in status:
        return 3
    elif "結婚" in status:
        return 2
    elif "独身" in status:
        return 1
    elif "未婚" in status:
        return 0
    else:
        print(f"Unknown status: {status}")
        return None

train_df['Marriage'] = train_df['Marriage'].apply(normalize_marriage_status)
test_df['Marriage'] = test_df['Marriage'].apply(normalize_marriage_status)

def normalize_car_ownership(ownership):
    if pd.isna(ownership):
        return None
    ownership = unicodedata.normalize('NFKC', ownership).lower()
    no_ownership_keywords = ["未所持", "未所有", "なし"]
    yes_ownership_keywords = ["所持", "所有", "あり", "保有"]

    if any(keyword in ownership for keyword in no_ownership_keywords):
        return 0
    elif any(keyword in ownership for keyword in yes_ownership_keywords):
        return 1
    else:
        print(f"Unknown ownership: {ownership}")
        return None

train_df['Own_Car'] = train_df['Own_Car'].apply(normalize_car_ownership)
test_df['Own_Car'] = test_df['Own_Car'].apply(normalize_car_ownership)

def normalize_children(children_str):
    if pd.isna(children_str):
        return 0

    children_str = unicodedata.normalize('NFKC', children_str).lower()

    # 数字を抽出
    numbers = re.findall(r'\d+', children_str)
    if numbers:
        return int(numbers[0])

    # 特定のキーワードに対応する数値を返す
    no_data_keywords = ["不明", "不詳", "わからない"]
    no_children_keywords = ["なし", "いない", "0人", "0", "ゼロ", "無", "非育児"]
    if any(keyword in children_str for keyword in no_children_keywords):
        return 0
    elif "一" in children_str or "1" in children_str:
        return 1
    elif "二" in children_str or "2" in children_str:
        return 2
    elif "三" in children_str or "3" in children_str:
        return 3
    elif "四" in children_str or "4" in children_str:
        return 4
    elif "五" in children_str or "5" in children_str:
        return 5
    elif any(keyword in children_str for keyword in no_data_keywords):
        return None
    # その他の場合は0を返す
    else:
        print(f"Unknown children status: {children_str}")
        return 0

train_df['Children'] = train_df['Children'].apply(normalize_children)
test_df['Children'] = test_df['Children'].apply(normalize_children)


# 6. カテゴリ変数のエンコーディング
for col in categorical_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col])
    # テストデータに未知のラベルがある場合を防ぐため、学習データのラベルに限定
    test_df[col] = test_df[col].map(lambda s: le.transform([s])[0] if s in le.classes_ else train_df[col].mode()[0])

train_df.drop(columns=["customer_info"], inplace=True)
test_df.drop(columns=["customer_info"], inplace=True)

# 7. 異常値の処理
train_df["MonthlyIncome"] = train_df["MonthlyIncome"].clip(lower=0, upper=train_df["MonthlyIncome"].quantile(0.99))
test_df["MonthlyIncome"] = test_df["MonthlyIncome"].clip(lower=0, upper=train_df["MonthlyIncome"].quantile(0.99))


# 前処理後のデータを確認
print("\nPreprocessed Train Data:")
print(train_df.head(50))

# 前処理済みデータを保存
train_df.to_csv("../data/train_preprocessed.csv", index=False)
test_df.to_csv("../data/test_preprocessed.csv", index=False)


Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown text: fe,male
Unknown te